##########################################
#                                        #
#                D(E)W(E)Y               #
#       IDK - Something DWI related      #
#                                        #
##########################################

"v7.yes Dewey from malcolm in the middle, MA"



### Version 25.07.2026 @ MA Python3.12 -- Linux version

DeweY expects folder structure like:
Whatever BIDS....
        dwi
            |Prep
            |Denoise 
            |Gibbs 
            etc...
 Each step has its own folder.
 Si nun te piace, amen stacce



In [64]:
"""
SET UP lib. and main flags
---------------------------------------------
DEBUGGING / LOGGING FLAGS
"""
import subprocess
import os
import sys
import numpy as np
import socket
import getpass
from datetime import datetime
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation, PillowWriter
import pandas as pd
import nibabel as nib
import glob as gl
import nilearn as nil
import shutil

import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from pathlib import Path

## DIPY 
from dipy.io.image import load_nifti, save_nifti
from dipy.core.gradients import gradient_table
from dipy.io.gradients import read_bvals_bvecs
import dipy.reconst.fwdti as fwdti
import dipy.reconst.dki as dki
import dipy.reconst.msdki as msdki ## I've never implemented this model because there are not enough info from litterature 

user=(getpass.getuser())
DEWEY_Version="DEWEY_v6"

# =====================================
# ==  SET UP 
# =====================================

### set path for fsl and freesurfer
fsl_path=r"/usr/local/fsl"

# ── Standard-space references ─────────────────────────────────────────────────
MNI            = f"{fsl_path}/data/standard/MNI152_T1_2mm.nii.gz"
MNI_BRAIN      = f"{fsl_path}/data/standard/MNI152_T1_2mm_brain.nii.gz"
MNI_MASK       = f"{fsl_path}/data/standard/MNI152_T1_2mm_brain_mask.nii.gz"


## SET WORKING DIRECTORY
prj_path=f"/kyb/rg/{user}/Unix_Folders/HARBOR/dewey_v7" # main BIDS path /kyb/rg/malberti/Unix_Folders/Sweep/Pilot/SW005  ## Calathealnx/browallnx --> /kyb/rg/malberti/Unix_Folders/SWEEP2/Script
project="sub-" #Common name for folders and files, for example all my sweep files are named as "sw001_ss1_dwi_AP.nii || sw001_ss1_T1w.nii"
script=f"/kyb/rg/{user}/Unix_Folders/SWEEP2/Script"
config_files=f"/kyb/rg/{user}/Unix_Folders/SWEEP2/Script/{DEWEY_Version}/config"
SUBJTAG="sub-1" 
HCPEX_BRAIN    = f"{script}/{DEWEY_Version}/Atlases/MNI_icbm_152_Template/tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz"
HCPEX_MASK     = f"{script}/{DEWEY_Version}/Atlases/MNI_icbm_152_Template/tpl-MNI152NLin2009cAsym_res-02_desc-brain_mask.nii.gz"

vps = [1] #0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43] #, 36, 37, 38, 39, 40, 41, 42, 43]
#vps = [3, 4, 12, 23, 31, 34, 40, 0, 1, 2, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 24, 25, 26, 27, 28, 29, 30, 33, 35, 36, 37, 38, 39, 41, 42, 43] #, 36, 37, 38, 39, 40, 41, 42, 43]

sessions=["ses-1" ,"ses-2"] #same with potatoes THE SCRIPT ASSUMES THAT SESSIONS FOLDER ARE NAMED AS 'ses-0*'  #Number 1, sess1 checl 

#These are the metric that the script can estimate - In future i will add more 
# Select oder of AP and PAs - Relatative to SWEEP2


BASH_SUFFIX="mk" # --> change name of bash files

do_prep=True # HIGHLY RECOMEND and study specific, otherwise the script doesn't (almost) work 
do_denoise=True #Run MP-PCA denoising, mandatory if number of volumes is higher that 32
do_gibbsring=True # you can run it by using a stand alone script after index estimation
do_topup=True # Just run it and shut up
do_eddy=True # Same with potatoes 
do_bias=True  #run ANTs N4biascorrection, almost useless (?) - not yet implemented
do_signaldrift=True #Run patch wise signal drift correction
do_coreg=True
do_smoothing=True

# Scalar maps
do_dtifit=True  # ---> Dtifit
do_kurtosis=True  
do_noddi=True
do_freewater=True


## MASKING & DENOISE setting ##
# if it sets to 1, every time that the script creates a mask, a while loop will start. 
# it allows the used to check bet mask and/or change bet f and g value to get the best mask as possible 
# Use this flag with asian (just a bit of racism sorry) or small brain

f=0.48  # base/standard f and g values 
g=0

phase_avail=1 # If phase image is available, please set it to 1 and the script will run denoising on complex data instead Mag. file only
phase_range=4096 #it is usually estimate by ((max+(-min))/2. Phase must be in radiants and the range is [-pi..pi] 


### Variables set up:
DataType=["mag", "phase"]

config= r"/usr/local/fsl/etc/flirtsch/b02b0.cnf" #FSL config file
acqparams=r"/home/malberti/Unix_Folders/HARBOR/dewey_v7/acqparams.txt"
index=r"/home/malberti/Unix_Folders/HARBOR/dewey_v7/index.txt"

## Generate bash script

In [65]:
bash_dir = os.path.join(script, 'DEWEY_v6', 'bash')
os.makedirs(bash_dir, exist_ok=True)
overwrite = False

for vp in vps: 
        for session in sessions: 
                id = f"{SUBJTAG}{vp:02d}"
                subjid = f"{id}_{session}"
               
                bash_script = os.path.join(bash_dir, f"{subjid}_{BASH_SUFFIX}.sh")
                #subprocess.run(f"rm -rf {bash_script}", shell=True) #Remove the old one
                
                # --- ENV --- setting
                ## Set Path & Steps settings

                standard_paths= f"FSLDIR={fsl_path}\n"
                # Templates

                ## SET WORKING DIRECTORY
                standard_paths+= f"prj_path={prj_path}\n"
                standard_paths+= f"script={script}\n"
                
                if not overwrite:
                        if not os.path.exists(bash_script):
                                overwrite = True
                        else:
                                confirm = input(f"File already exists: {bash_script}\nOverwrite? (y/n): ")
                                overwrite = True

                if overwrite:
                        with open(bash_script, "w") as f:
                                f.write("#!/bin/bash\n\n")
                                f.write("###############\n")
                                f.write("## DEWEY v06 ##\n")
                                f.write("###############\n\n")
                                f.write(f"# User    : {getpass.getuser()}\n")
                                f.write(f"# Host    : {socket.gethostname()}\n")
                                f.write(f"# IP      : {socket.gethostbyname(socket.gethostname())}\n")
                                f.write(f"# Created : {datetime.now()}\n\n")
                                f.write(standard_paths)
                        print(f"Written: {bash_script}")
                else:
                        exit

                                
                subprocess.run(f"chmod +x {bash_script}", shell=True)
# Initialize LOGs file
for vp in vps:
        for session in sessions:    # Loop over sessions for this participant
            id = f"{SUBJTAG}{vp:02d}"   # id: standard subject ID (sub-XX)
            subjid = f"{id}_{session}"     # Standardized subject-session identifier 
            LOGs_dir =os.path.join(script, "LOGs")
            
            # ============================================================================
            ## Start LOGs file
            os.makedirs(os.path.join(LOGs_dir, f"{subjid}-LOGs"), exist_ok=True)
            timestamp = datetime.now().strftime("%H%M%d%m%Y")

            LOGsOut = os.path.join(
                LOGs_dir,
                f"{subjid}-LOGs",
                f"{subjid}-LOGs_{timestamp}.txt"
            )

            with open(LOGsOut, "w") as f:
                f.write("#########################\n")
                f.write("## DEWEY preprocessing ##\n")
                f.write("#########################\n\n")
                f.write(f"{getpass.getuser()}\n")
                f.write(f"{socket.gethostname()}\n")
                f.write(f"{socket.gethostbyname(socket.gethostname())}\n")
                f.write(f"Subject BIDS ID: {id}, {session}\n\n\n")                          

Written: /kyb/rg/malberti/Unix_Folders/SWEEP2/Script/DEWEY_v6/bash/sub-101_ses-1_mk.sh
Written: /kyb/rg/malberti/Unix_Folders/SWEEP2/Script/DEWEY_v6/bash/sub-101_ses-2_mk.sh


## RAWDATA setting up
-----------------------------------------
 Full sequence Denoising: We run denoising on the entire sequence then i split again for topup and eddy current 

In [59]:
"""
The first block 'Setting up files...' it is study specific and hard coded, you need to rearrange/change it based on you data set... The output, must be a seq that looks like: 
PA --> AP --> PA 
+ bvecs and bvals files
"""

NdoMinchiaSiamo="Setting up files" #Define the step on the log file
os.chdir(os.path.join(script, 'DEWEY_v6'))

command=f"\n"
bash_dir = os.path.join(script, 'DEWEY_v6', 'bash')
os.makedirs(bash_dir, exist_ok=True)

if do_prep:
   for vp in vps:
        for session in sessions:    # Loop over sessions for this participant
            id = f"{SUBJTAG}{vp:02d}"   # id: standard subject ID (sub-XX)
            subjid = f"{id}_{session}"     # Standardized subject-session identifier 
            bash_script = os.path.join(bash_dir, f"{subjid}_{BASH_SUFFIX}.sh")            
            LOGsOut = os.path.join(LOGs_dir,f"{subjid}-LOGs",f"{subjid}-LOGs_{timestamp}.txt")
            print(f"Compiling: {bash_script}")

            orig = os.path.join(prj_path, "rawdata", id, session)
            derivatives = os.path.join(prj_path, "derivatives", id, session)
            dwi_prep = os.path.join(derivatives, "dwi", "prep")  # Step path
            
            # ============================================================================

            command+=f"echo '=================================================================='\n"
            command+=f"echo 'Starting DWI prep... (merging/splitting/checking images): {subjid}'\n"
            command+=f"echo '=================================================================='\n"
            
            command+=f"orig={orig}\n"
            command+=f"derivatives={derivatives}\n\n"
            command+=f"dwi_prep={dwi_prep}\n\n"

            command+= f'if [ ! -d "{derivatives}" ]; then mkdir -p "{derivatives}"; fi\n'
            command+= f'if [ ! -d "{dwi_prep}" ]; then mkdir -p "{dwi_prep}"; fi\n'
    
            command+=f"echo 'Preparing PAs b0s....: {subjid}'\n"
            
            for data in DataType: #loop across phase and magnitude
                ## Run PAs/APs preparation                
                # --- Extract B0 volumes from each PA ---------------------------------------------------------------------------------------------------------------
                # Iterate over PAs
                for i in range (1,3): #Handle PAs.
                    prefix_in=os.path.join(orig, 'dwi',f"{subjid}_dir-PA_run-{i}_part-{data}_dwi") #Pick the first PAs image  <-------
                    prefix_out=os.path.join(dwi_prep,f"{subjid}_dir-PA_run-{i}_part-{data}_dwi")
                    
                    DWIs_in=os.path.join(f"{prefix_in}.nii.gz")
                    BVECs_in=os.path.join(f"{prefix_in}.bvec")
                    BVALs_in=os.path.join(f"{prefix_in}.bval")
                    
                    DWIs_out=os.path.join(f"{prefix_out}.nii.gz")
                    BVECs_out=os.path.join(f"{prefix_out}.bvec")
                    BVALs_out=os.path.join(f"{prefix_out}.bval")

                    command+=f"dwiextract {DWIs_in} -bzero {DWIs_out} -fslgrad {BVECs_in} {BVALs_in} -export_grad_fsl {BVECs_out} {BVALs_out} -force -debug \n"
                    command+=f"echo 'PAs B0s volumes check: {DWIs_out}'  >> {LOGsOut} \n" 
                    command+=f"mrstats {DWIs_out}>> {LOGsOut}  \n "  
            
                command+=f"echo 'Merging APs and PAs....: {subjid}'\n"

                DWIs=os.path.join(dwi_prep,f"{subjid}_dir-PA-AP_part-{data}_dwi.nii.gz")
                DWIs_JustToBeSafe=os.path.join(dwi_prep,f"{subjid}_dir-PA-AP_part-{data}_dwi.mif")

                BVECs=os.path.join(dwi_prep,f"{subjid}_dir-PA-AP_part-{data}_dwi.bvec")
                BVALs=os.path.join(dwi_prep,f"{subjid}_dir-PA-AP_part-{data}_dwi.bval")

                command+=f"mrcat {os.path.join(dwi_prep,f"{subjid}_dir-PA_run-1_part-{data}_dwi.nii.gz")} {os.path.join(orig, 'dwi',f"{subjid}_dir-AP_part-{data}_dwi.nii.gz")} {os.path.join(dwi_prep,f"{subjid}_dir-PA_run-2_part-{data}_dwi.nii.gz")} {DWIs} -force -debug \n"

                # Merge bval and bvecs -- \nSometimes it fails, check it later during EDDY 
                # Before i remove the bvec/bvals files if they exists, to avoid any issue 
                # keep thema s 2 diff. if statement, usually is better 
                # 'Paste' joins files horizontally, output: lines of lines form each file
                
                command+=f"rm -rf {BVECs} \n"
                command+=f"rm -rf {BVALs} \n"
                command+=f"echo 'Merging BVAls and BVECs....: {subjid}'\n"
                command+=f"paste -d ' ' {os.path.join(dwi_prep,f"{subjid}_dir-PA_run-1_part-{data}_dwi.bvec")} {os.path.join(orig, 'dwi',f"{subjid}_dir-AP_part-{data}_dwi.bvec")} {os.path.join(dwi_prep,f"{subjid}_dir-PA_run-2_part-{data}_dwi.bvec")} > {BVECs} \n"
                command+=f"paste -d ' ' {os.path.join(dwi_prep,f"{subjid}_dir-PA_run-1_part-{data}_dwi.bval")} {os.path.join(orig, 'dwi',f"{subjid}_dir-AP_part-{data}_dwi.bval")} {os.path.join(dwi_prep,f"{subjid}_dir-PA_run-2_part-{data}_dwi.bval")} > {BVALs} \n"
                command+=f"cat {BVALs} >> {LOGsOut} \n"
                command+=f"cat {BVECs} >> {LOGsOut} \n"

                command+=f"mrconvert {DWIs} {DWIs_JustToBeSafe} -fslgrad {BVECs} {BVALs} -force -debug\n" ## I create this mif file because if there are issues on bvecs and bvals mrtrix3 will complain and tell u what is not allign 
                command+=f"rm  {DWIs_JustToBeSafe}\n" #No errors
                ## Quality controll and slices dire output: 
                DWIs_PNG=os.path.join(dwi_prep,f"{subjid}_dir-PA-AP_part-{data}_dwi")
                command+=f"python {script}/DEWEY_v6/HomeMadeSlicesDir.py -i {DWIs} -bval {BVALs} -bvec {BVECs} -out {dwi_prep} -outpng {DWIs_PNG}\n\n "
                command+=f"echo 'Preparation completed: Main output values---> {subjid} >> {LOGsOut} ' \n"
                command+=f"mrstats {DWIs} >> {LOGsOut}"   

                with open(bash_script, "a") as f: #Update LOGs file
                        f.write(command)
                command=f"\n"

Compiling: /kyb/rg/malberti/Unix_Folders/SWEEP2/Script/DEWEY_v6/bash/sub-101_ses-1_mk.sh
Compiling: /kyb/rg/malberti/Unix_Folders/SWEEP2/Script/DEWEY_v6/bash/sub-101_ses-2_mk.sh


## Denoising
-----------------------

The denoising is performed by using MP-PCA algorithm on complex data. 
DEWEI will run the denosing on each, individually AP block in order to avoid issues and mistakes given by
motion artifacts due to saliva sampling and blocks
we will run denoising on: 
1st PA + 1st AP 
2ndAP 
3trAP + 4th PA 
in the end, the script merge everything together and prepare the sequence to run topup correction on denoised B0s

Phase in radiant and complex data will be obtained by using mrcalc, it apply generic voxel-wise mathematical operations to images
USAGE
mrcalc [ options ] operand [ operand ... ]

operand an input image, intensity value, or the special keywords
        'rand' (random number between 0 and 1) or 'randn' (random
        number from unit std.dev. normal distribution) or the
        mathematical constants 'e' and 'pi'.

DESCRIPTION
This command will only compute per-voxel operations. Use 'mrmath' to compute summary statistics across images or along image axes.
This command uses a stack-based syntax, with operators (specified using options) operating on the top-most entries (i.e. images or values) in the
stack. Operands (values or images) are pushed onto the stack in the order they appear (as arguments) on the command-line, and operators (specified as options) operate on and consume the top-most entries in the stack, and push their output as a new entry on the stack.
As an additional feature, this command will allow images with different dimensions to be processed, provided they satisfy the following conditions: for each axis, the dimensions match if they are the same size, or one of them has size one. In the latter case, the entire image will be
replicated along that axis. This allows for example a 4D image of size [ X Y Z N ] to be added to a 3D image of size [ X Y Z ], as if it consisted of N copies of the 3D image along the 4th axis (the missing dimension is assumed to have size 1). Another example would a single-voxel 4D image of
size [ 1 1 1 N ], multiplied by a 3D image of size [ X Y Z ], which would allow the creation of a 4D image where each volume consists of the 3D image scaled by the corresponding value for that volume in the single-voxel image.

    
The denoise bash script takes as input the common filename pattern
    ({subjid}_dir-PA-AP_part)
which is shared across magnitude, phase, and complex images.

The script also requires:
    - an input folder
    - an output folder

All other naming conventions and processing steps are fully automatic and hard-coded in the script.

Output files (all saved in {denoise_path}):

1) Radiant phase image
]]{subjid}_dir-PA-AP_part-rad_dwi.nii.gz
Phase image converted to radians.

2) Complex image
{subjid}_dir-PA-AP_part-complex_dwi.nii.gz
Complex-valued image obtained by combining magnitude and radiant phase.

3) Denoised magnitude image
{subjid}_dir-PA-AP_part-mag_dwi_denoised.nii.gz
Denoised DWI using MPPCA.

4) Noise estimate
{subjid}_dir-PA-AP_part-mag_dwi_noise.nii.gz
Noise kernel estimated via mean MPPCA.

5) Residual image
{subjid}_dir-PA-AP_part-mag_dwi_residue.nii.gz
Difference between the raw and denoised magnitude images.

To add or remove specific flags in the dwidenoise call,
please edit the DEWEY_denoise script.



------------------------------------------------------------------------

The script below comprises also noise rectification - it is structured as a stand-alone python script

In [60]:
command=f"\n"

if do_denoise:
   
    NdoMinchiaSiamo="Denoising"
    
    for vp in vps:
            for session in sessions:    # Loop over sessions for this participant
                id = f"{SUBJTAG}{vp:02d}"   # id: standard subject ID (sub-XX)
                subjid = f"{id}_{session}"     # Standardized subject-session identifier 
                bash_script = os.path.join(bash_dir, f"{subjid}_{BASH_SUFFIX}.sh")
                print(f"Compiling: {bash_script}")
                
                derivatives = os.path.join(prj_path, "derivatives", id, session)
                dwi_prep = os.path.join(derivatives, "dwi", "prep")  # Step path
                denoise_path=os.path.join(derivatives, "dwi","denoise")
                LOGsOut = os.path.join(LOGs_dir, f"{subjid}-LOGs", f"{subjid}-LOGs_{timestamp}.txt")
               
               # =============================================================================

                command+=f"echo '======================================='\n"
                command+=f"echo 'Starting DWI MP-PCA denoising: {subjid}'\n"
                command+=f"echo '======================================='\n\n"

                # FIle in path define 
                DWIs_mag=os.path.join(dwi_prep,f"{subjid}_dir-PA-AP_part-mag_dwi.nii.gz")
                DWIs_phase=os.path.join(dwi_prep,f"{subjid}_dir-PA-AP_part-phase_dwi.nii.gz")
                
                BVECs=os.path.join(dwi_prep,f"{subjid}_dir-PA-AP_part-mag_dwi.bvec")
                BVALs=os.path.join(dwi_prep,f"{subjid}_dir-PA-AP_part-mag_dwi.bval")

              #  DWIs_complex=os.path.join(denoise_path,f"{subjid}_dir-PA-AP_part-complex_dwi.nii.gz")
              #  DWIs_phase_rad=os.path.join(denoise_path,f"{subjid}_dir-PA-AP_part-phase-rad_dwi.nii.gz")

                DWIs_out_prefix=os.path.join(denoise_path,f"{subjid}_dir-PA-AP_part-mag_dwi")
                DWIs_noise=os.path.join(denoise_path,f"{DWIs_out_prefix}_noise.nii.gz")
                DWIs_sqr_weighted_noise=os.path.join(denoise_path,f"{DWIs_out_prefix}_sqr_wei_noise.nii.gz")
                DWIs_residue=os.path.join(denoise_path,f"{DWIs_out_prefix}_residue.nii.gz")
                DWIs_denoised=os.path.join(denoise_path,f"{DWIs_out_prefix}_denoised.nii.gz")

                #if  os.path.exists(DWIs_phase):   #The script check that phase ecxists, in case it doesn't exist it runs denoising on magnitude only 
                command+= f'if [ ! -d "{denoise_path}" ]; then mkdir -p "{denoise_path}"; fi\n'
                command+=f"python MP-PCA_denoise_Mrtrix3.py -i {DWIs_mag} -p {DWIs_phase} --bval {BVALs} --bvec {BVECs} -o {DWIs_out_prefix} --do_rect \n"
                
                ## Quality controll and slices dire output: 
                command+=f"fslmaths {DWIs_mag} -sub {DWIs_denoised} {DWIs_residue} \n" #Get residue 
                command+=f"fslmaths {DWIs_noise} -sqrt {DWIs_sqr_weighted_noise}\n"
                command+=f"fslmaths {DWIs_residue} -div {DWIs_sqr_weighted_noise} {DWIs_sqr_weighted_noise}\n" # QA Output
                
                DWIs_PNG=os.path.join(denoise_path,f"{subjid}_dir-PA-AP_part-mag_dwi_denoised_ABC")
                command+=f"python {script}/DEWEY_v6/HomeMadeSlicesDir.py -i {DWIs_denoised} -bval {BVALs} -bvec {BVECs} -out {denoise_path} -outpng {DWIs_PNG}\n\n " #Denoised image
                
                DWIs_PNG=os.path.join(denoise_path,f"{subjid}_dir-PA-AP_part-noise_dwi_ABC")
                command+=f"python {script}/DEWEY_v6/HomeMadeSlicesDir.py -i {DWIs_noise} -out {denoise_path} -outpng {DWIs_PNG}\n\n " #Noise
                
                DWIs_PNG=os.path.join(denoise_path,f"{subjid}_dir-PA-AP_part-sqr_wei_noise_dwi_ABC")
                command+=f"python {script}/DEWEY_v6/HomeMadeSlicesDir.py -i {DWIs_sqr_weighted_noise} -bval {BVALs} -bvec {BVECs}  -out {denoise_path} -outpng {DWIs_PNG}\n\n " #Noise             
                
                command+=f"echo '\n\nDenoising completed: Main output values---> {subjid} >> {LOGsOut} ' \n"
                command+=f"mrstats {DWIs_denoised} >> {LOGsOut}\n" 

                ## Clean-up the folder and remove intermediate files:  aka complex, phase etc...
                command+=f"rm {DWIs_sqr_weighted_noise}\n"

                with open(bash_script, "a") as f: #Update LOGs file
                    f.write(command)
                command=f"\n"

Compiling: /kyb/rg/malberti/Unix_Folders/SWEEP2/Script/DEWEY_v6/bash/sub-101_ses-1_mk.sh
Compiling: /kyb/rg/malberti/Unix_Folders/SWEEP2/Script/DEWEY_v6/bash/sub-101_ses-2_mk.sh


## GIBBS ring correction
----------------------------
The algorithm uses mrtrix3 Gibbs ring correction, it perform the correction on the denoised image if it exits, otherwise on the original mag. 

In [61]:
command=f"\n"

if do_gibbsring == 1:
   
    for vp in vps:
            for session in sessions:    # Loop over sessions for this participant
                id = f"{SUBJTAG}{vp:02d}"   # id: standard subject ID (sub-XX)
                subjid = f"{id}_{session}"     # Standardized subject-session identifier 
                bash_script = os.path.join(bash_dir, f"{subjid}_{BASH_SUFFIX}.sh")

                print(f"Compiling: {bash_script}")
                
                derivatives = os.path.join(prj_path, "derivatives", id, session)
                dwi_prep = os.path.join(derivatives, "dwi", "prep")  # Step path
                denoise_path=os.path.join(derivatives, "dwi","denoise")
                gibbs_path=os.path.join(derivatives, "dwi","gibbs_ring")
           
                # ============================================================================
                ## Start LOGs file
                LOGsOut = os.path.join(LOGs_dir, f"{subjid}-LOGs", f"{subjid}-LOGs_{timestamp}.txt")
                       
                # Gibbs ring  Input and Output
                
                DWIs_mag=os.path.join(dwi_prep,f"{subjid}_dir-PA-AP_part-mag_dwi.nii.gz")
                DWIs_denoised=os.path.join(denoise_path,f"{subjid}_dir-PA-AP_part-mag_dwi_denoised.nii.gz")

                if do_denoise:
                    DWIs_out_prefix=os.path.join(gibbs_path,f"{subjid}_dir-PA-AP_part-mag_dwi_denoised")
                    in_DwIs=DWIs_denoised
                else: 
                    DWIs_out_prefix=os.path.join(gibbs_path,f"{subjid}_dir-PA-AP_part-mag_dwi")
                    in_DwIs=DWIs_mag
                
                DWIs_residue=os.path.join(gibbs_path, f"{DWIs_out_prefix}_residue.nii*")   

                command+=f"echo '======================================='\n"
                command+=f"echo 'Starting Gibb ring correction: {subjid}'\n"
                command+=f"echo '======================================='\n"
                command+=f"echo ' Input DWIs       : {in_DwIs}'\n"
                command+=f"echo ' Residue DWIs     : {DWIs_residue}'\n"
                command+=f"echo ' Output DWIs      : {DWIs_out_prefix}_degibbs.nii.gz'\n"
                command+=f"echo '======================================='\n\n"

                # ---------------------------
                # Run degibbsing
                # MRtrix3 mrdegibbs
                # Applies sub-voxel shifts to reduce Gibbs ringing
                # --------------------------------------------------
                command+= f'if [ ! -d "{gibbs_path}" ]; then mkdir -p "{gibbs_path}"; fi\n'
                command+=f"mrdegibbs {in_DwIs} {DWIs_out_prefix}_degibbs.nii.gz -force -debug \n"

                # --------------------------------------------------
                # Compute residual image: original - degibbsed
                # --------------------------------------------------
                command+=f"fslmaths {in_DwIs} -sub {DWIs_out_prefix}_degibbs.nii.gz {DWIs_residue}\n"

                DWIs_PNG=os.path.join(gibbs_path,f"{subjid}_dir-PA-AP_residue")
                command+=f"python {script}/DEWEY_v6/HomeMadeSlicesDir.py -i {DWIs_residue} -out {gibbs_path} -outpng {DWIs_PNG}\n " #Noise
                command+=f"echo '\n\nGibbs ring completed: Main output values---> {subjid} >> {LOGsOut} ' \n"
                command+=f"mrstats {DWIs_out_prefix}_degibbs.nii.gz >> {LOGsOut}\n"  
                

                with open(bash_script, "a") as f: #Update LOGs file
                    f.write(command)
                command=f"\n"


Compiling: /kyb/rg/malberti/Unix_Folders/SWEEP2/Script/DEWEY_v6/bash/sub-101_ses-1_mk.sh
Compiling: /kyb/rg/malberti/Unix_Folders/SWEEP2/Script/DEWEY_v6/bash/sub-101_ses-2_mk.sh


## TOPUP 
----------------------------
Performerd via FSL topup, the correction require quite a lot of time, probably a couple of hourse on 300+ volumes DWIs,
It also require acqparams file, which are different within sweep data frame, so please check the acq/index selections below  

 """
    Topup Correction (FSL)

    --- Description ---
    Topup is used to correct for **susceptibility-induced distortions** in diffusion MRI.
    It estimates the spatial field inhomogeneity (in Hz) caused by magnetic susceptibility,
    then applies a correction to the images. Typically, it is applied to the b0 volumes
    of DWI data acquired with opposite phase-encoding directions (e.g., PA and AP).

    --- Compulsory arguments ---
    You MUST provide at least one of the following:
        --imain      : 4D file containing the images to correct (usually extracted b0s)

    --- Optional arguments ---
        --acqp       : Text file specifying phase-encoding directions and readout times
        --out        : Base name for output spline coefficients (Hz) and movement parameters
        --fout       : Name of image file with estimated field (Hz)
        --iout       : Name of 4D image file with unwarped images
        --featout    : Base name for exporting results to FEAT
        --nthr       : Number of threads to use (default = 1; cannot exceed hardware cores)
        -h, --help   : Display help information
        -v, --verbose: Print diagnostic information during processing

    --- How to run Topup ---
    1. Extract all b0 images from your main DWI dataset.
    2. Set up the acquisition parameters in a text file. Example format:
        0  1  0  [total readout time]   # for PA
        0 -1  0  [total readout time]   # for AP
    3. Run topup using the extracted b0s and the acquisition parameters.

    --- References ---
    More info: https://fsl.fmrib.ox.ac.uk/fsl/fslwiki/EDDY/Faq#Topup
    
"""

In [ ]:
# TopUP ----------------------------
command=f"\n"

if do_topup == 1:
    NdoMinchiaSiamo="TOPUP"
   
    for vp in vps:
        id = f"{SUBJTAG}{vp:02d}"   # id: standard subject ID (sub-XX)
        for session in sessions:    # Loop over sessions for this participant
            subjid = f"{id}_{session}"     # Standardized subject-session identifier 
            bash_script = os.path.join(bash_dir, f"{subjid}_{BASH_SUFFIX}.sh")

            derivatives = os.path.join(prj_path, "derivatives", id, session)
            print(f"Compiling: {bash_script}")
            
            LOGsOut = os.path.join(LOGs_dir, f"{subjid}-LOGs", f"{subjid}-LOGs_{timestamp}.txt")
            dwi_prep = os.path.join(derivatives, "dwi", "prep")  # Step path                
            denoise_path=os.path.join(derivatives, "dwi","denoise") # Prep folder
            gibbs_path=os.path.join(derivatives, "dwi","gibbs_ring")
            topup_path=os.path.join(derivatives, "dwi","topup")
            
            command+= f'if [ ! -d "{topup_path}" ]; then mkdir -p "{topup_path}"; fi\n'
            in_BVECs = os.path.join(dwi_prep,f"{subjid}_dir-PA-AP_part-mag_dwi.bvec")
            in_BVALs = os.path.join(dwi_prep,f"{subjid}_dir-PA-AP_part-mag_dwi.bval")

            if do_gibbsring:
                in_DWIs = os.path.join(gibbs_path, f"{subjid}_dir-PA-AP_part-mag_dwi_*_degibbs.nii.gz")
            else:
                if do_denoise:
                    in_DWIs = os.path.join(denoise_path, f"{subjid}_dir-PA-AP_part-mag_dwi_denoised.nii.gz")
                else:
                    in_DWIs = os.path.join(dwi_prep, f"{subjid}_dir-PA-AP_part-mag_dwi.nii.gz")
                
            b0s=os.path.join(topup_path, f"{subjid}_dir-PA-AP_part-mag_dwi_b0s.nii.gz") #This file will contain all the b0s and it will be the input for topup
        
            topup_out=os.path.join(topup_path, f"topup_out")  #topup out common name
            topup_out_field=os.path.join(topup_path, f"topup_out_rfield") 
            topup_out_warp=os.path.join(topup_path, f"topup_out_unwarped") 

            command+=f"echo '======================================='\n"
            command+=f"echo 'Topup:  {subjid}'\n"
            command+=f"echo '======================================='\n"
            command+=f"echo ' Input DWIs       : {in_DWIs}'\n"
            command+=f"echo ' Input B0s        : {b0s}'\n"
            command+=f"echo '======================================='\n"

            # Extract b0 images using FSL's dwiextract
            # -bzero : extract volumes with b=0
            # -fslgrad : specify gradient files (bvecs/bvals)
            command+=f"dwiextract {in_DWIs} -bzero {b0s} -fslgrad {in_BVECs} {in_BVALs} -force -debug \n"  
            command+=f"topup --imain={b0s}  --datain={acqparams} --config={config} --out={topup_out} --fout={topup_out_field} --iout={topup_out_warp} --verbose --nthr=24 \n"
            
            # Quality controll 
            command+=f"python {script}/DEWEY_v6/RMS_movement_plots.py -i {topup_out}_movpar.txt -o {topup_path} -subjid {subjid} \n" 
            command+=f"echo 'Topup Completet: Unwarped B0s values---> {subjid} ' >> {LOGsOut}\n "
            command+=f"mrstats {topup_out_warp}.nii >> {LOGsOut}\n  "
            
            command+=f"rm {b0s}\n"

            with open(bash_script, "a") as f: #Update LOGs file
                f.write(command)
            command=f"\n"

## EDDY current correction & Motion cofrection 
-------------------------------------------------------

In [ ]:
# EDDY & movement correction ----------------------------
command=f"\n"
import glob
if do_eddy == 1:
    for vp in vps:
        id = f"{SUBJTAG}{vp:02d}"   # id: standard subject ID (sub-XX)
        for session in sessions:    # Loop over sessions for this participant
            subjid = f"{id}_{session}"     # Standardized subject-session identifier 
            bash_script = os.path.join(bash_dir, f"{subjid}_{BASH_SUFFIX}.sh")
            print(f"Compiling: {bash_script}")
            
            derivatives = os.path.join(prj_path, "derivatives", id, session)  #
            LOGsOut = os.path.join(LOGs_dir, f"{subjid}-LOGs", f"{subjid}-LOGs_{timestamp}.txt")
            
            dwi_prep = os.path.join(derivatives, "dwi", "prep")  # Step path                
            denoise_path=os.path.join(derivatives, "dwi","denoise") # Prep folder
            gibbs_path=os.path.join(derivatives, "dwi","gibbs_ring")
            topup_path=os.path.join(derivatives, "dwi","topup")
            eddy_path=os.path.join(derivatives, "dwi","eddy")

            if os.path.exists(eddy_path): 
                subprocess.run(f"rm -rf {eddy_path}", shell=True)
            
            os.makedirs(eddy_path, exist_ok=True)

            command+= f'if [ ! -d "{eddy_path}" ]; then mkdir -p "{eddy_path}"; fi\n'

            eddy_out=os.path.join(eddy_path,f"{subjid}_dwi_eddy_corrected") 
                                    
            in_BVECs = os.path.join(dwi_prep,f"{subjid}_dir-PA-AP_part-mag_dwi.bvec")
            in_BVALs = os.path.join(dwi_prep,f"{subjid}_dir-PA-AP_part-mag_dwi.bval")

            if do_gibbsring:
                in_DWIs = os.path.join(gibbs_path, f"{subjid}_dir-PA-AP_part-mag_dwi_denoised_degibbs.nii.gz")
            elif do_denoise:
                in_DWIs = os.path.join(denoise_path, f"{subjid}_dir-PA-AP_part-mag_dwi_denoised.nii.gz")
            
            else:
                in_DWIs = os.path.join(dwi_prep, f"{subjid}_dir-PA-AP_part-mag_dwi.nii.gz")
            
            topup_out=os.path.join(topup_path, f"topup_out")  #topup out common name
            topup_out_warp=os.path.join(topup_path, f"topup_out_unwarped_mean.nii") 
            mask_prefix=os.path.join(eddy_path, f"topup_out_unwarped_brain")
            mask=os.path.join(eddy_path, f"topup_out_unwarped_brain_mask.nii")
            

            command+=f"bet {topup_out_warp} {mask_prefix} -R -f 0.48 -g 0 -m \n"
            
            command+=f"echo '=================================================='\n"
            command+=f"echo 'Eddy current & motion correction:  {subjid}'\n"
            command+=f"echo '=================================================='\n"
            command+=f"echo ' Input DWIs         : {in_DWIs}'\n"
            command+=f"echo ' nput BVECs         : {in_BVECs}'\n"
            command+=f"echo ' Input BVALs        : {in_BVALs}'\n"
            command+=f"echo ' Outpup DWIs        : {eddy_out}'\n"
            command+=f"echo '=================================================='\n"

            # Extract b0 images using FSL's dwiextract
            # -bzero : extract volumes with b=0
            # -fslgrad : specify gradient files (bvecs/bvals)
            cmd = (
                            f"eddy_cuda10.2 "
                            f"--imain={in_DWIs} "
                            f"--mask={mask} "
                            f"--acqp={acqparams} "
                            f"--index={index} "
                            f"--bvecs={in_BVECs} "
                            f"--bvals={in_BVALs} "
                            f"--topup={topup_out} "
                            f"--out={eddy_out} "
                            f"--flm=quadratic "
                            f"--slm=quadratic " #quadratic or linear
                            f"--dont_peas "
                            f"--residuals "
                            f"--verbose "
                            f"--mb=2 "
                            f"--mporder=1 "
                            f"--cnr_maps "
                            f"--residuals "
                            f"--estimate_move_by_susceptibility "
                        )      
            """ 
            Additional flags: 
                ->      --data_is_shelled   Assume, don't check, that data is shelled (default false) Should be used if there are low directions b-shells            
                ->      --b_range           Range of b-values considered to be the same shell (default 100) -- Use it if you are processing IVIM data
                ->      --repol             Outlier slices 
                ->      --ol_nstd           Number of std off to qualify as outlier (default 4)
                ->      --ol_nvox           Min # of voxels in a slice for inclusion in outlier detection (default 250)
                ->      --ol_type           Type of outliers, slicewise (sw), groupwise (gw) or both (both). (default sw)
                ->      --ol_pos            Consider both positive and negative outliers if set (default false)
                ->      --ol_sqr            Consider outliers among sums-of-squared differences if set (default false)
                ->      --mb                Multi-band factor
                ->      --mporder=n         intravolume moco
                ->      --field_mat         Name of rigid body transform for susceptibility field --> maybe imrpove registrataion? 
            """
            command+=f"{cmd} \n"
            # Quality controll 
            command+=f"python {script}/DEWEY_v6/RMS_movement_plots.py -i {eddy_out}.eddy_parameters -o {topup_path} -subjid {subjid} \n" 
            command+=f"python {script}/DEWEY_v6/EDDY_RMS.py -i {eddy_out}.eddy_restricted_movement_rms -o {eddy_out} -subjid {subjid} \n" 
            command+=f"echo 'Eddy Completet: Eddy corrected values---> {subjid} ' >> {LOGsOut}\n "
            command+=f"mrstats {eddy_out}.nii >> {LOGsOut}\n  "

            with open(bash_script, "a") as f: #Update LOGs file
                f.write(command)
            command=f"\n"

## SIGNAL DRIFT correction 
-----------------------------------
It uses a code i personally i don't like, but whatever, here are more info - 
ps. the code has been comment and opimized by claude ai 

In [ ]:
command=f"\n"
do_signaldrift=1
if do_signaldrift==1: 
      for vp in vps:
        id = f"{SUBJTAG}{vp:02d}"   # id: standard subject ID (sub-XX)
        for session in sessions:    # Loop over sessions for this participant
            subjid = f"{id}_{session}"     # Standardized subject-session identifier 
            bash_script = os.path.join(bash_dir, f"{subjid}_{BASH_SUFFIX}.sh")
            print(f"Compiling: {bash_script}")
            derivatives = os.path.join(prj_path, "derivatives", id, session)  #
            LOGsOut = os.path.join(LOGs_dir, f"{subjid}-LOGs", f"{subjid}-LOGs_{timestamp}.txt")

            dwi_prep = os.path.join(derivatives, "dwi", "prep")  # Step path                
            denoise_path=os.path.join(derivatives, "dwi","denoise") # Prep folder
            gibbs_path=os.path.join(derivatives, "dwi","gibbs_ring")
            topup_path=os.path.join(derivatives, "dwi","topup")
            eddy_path=os.path.join(derivatives, "dwi","eddy")
            signal_drift=os.path.join(derivatives, "dwi","signal_drift")
            command+= f'if [ ! -d "{signal_drift}" ]; then mkdir -p "{signal_drift}"; fi\n'


            in_DWIs  = os.path.join(eddy_path,f"{subjid}_dwi_eddy_corrected.nii.gz")
            in_BVECs = os.path.join(eddy_path,f"{subjid}_dwi_eddy_corrected.eddy_rotated_bvecs")
            in_BVALs = os.path.join(dwi_prep,f"{subjid}_dir-PA-AP_part-mag_dwi.bval")
            
            mask=os.path.join(eddy_path, f"topup_out_unwarped_brain_mask.nii")
            
            DWIs_basename=os.path.join(signal_drift,f"{subjid}_eddy-current_signal-drift")


            command+=f"echo '=================================================='\n"
            command+=f"echo 'B0s signal drift:  {subjid} '\n"
            command+=f"echo '=================================================='\n"
            command+=f"echo ' Input DWIs         : {in_DWIs}'\n"
            command+=f"echo ' Input BVECs        : {in_BVECs}'\n"
            command+=f"echo ' Input BVALs        : {in_BVALs}'\n"
            command+=f"echo ' Outpup DWIs        : {DWIs_basename}'\n"
            command+=f"echo '=================================================='\n"

            #Run signal drift
            command+=f"mrconvert {in_DWIs} -coord 3 6:end -fslgrad {in_BVECs} {in_BVALs} {in_DWIs} -export_grad_fsl {in_BVECs} {in_BVALs} -force -debug \n"     ## remove PAs image
            command+=f"python signal_drift.py -i {in_DWIs} -bval {in_BVALs} -o {DWIs_basename} -roi {mask} \n"
            
            ## Quality controll and slices dire output: 
            DWIs_PNG=os.path.join(signal_drift,f"{subjid}_eddy-current_signal-drift_corr.png")
            command+=f"python {script}/DEWEY_v6/HomeMadeSlicesDir.py -i {os.path.join(signal_drift,f"{subjid}_eddy-current_signal-drift_corr.nii.gz")} -bval {in_BVALs} -bvec {in_BVECs} -out {signal_drift} -outpng {DWIs_PNG}\n\n " #Denoised image
            
            DWIs_PNG=os.path.join(signal_drift,f"{subjid}_eddy-current_signal-drift_resid.png")
            command+=f"python {script}/DEWEY_v6/HomeMadeSlicesDir.py -i {os.path.join(signal_drift,f"{subjid}_eddy-current_signal-drift_{AP}_corrfield.nii.gz")} -out {signal_drift} -outpng {DWIs_PNG}\n\n " #Noise
            
            command+=f"echo 'Signal drift completed: Main output values---> {subjid} >> {LOGsOut} ' \n"
            command+=f"echo 'Signal Drift Completet: Signal drift corrected values---> {subjid} ' >> {LOGsOut}\n "
            
            command+=f"mrstats {os.path.join(signal_drift,f"{subjid}_eddy-current_signal-drift_corr.nii.gz")} >> {LOGsOut} \n"      

            with open(bash_script, "a") as f: #Update LOGs file
                f.write(command)
            command=f"\n"

## BIAS field correction
-----------------------------------
It uses N4Ants bias field correction implemented on MRtrix3 - Almost useless if u don't go for tractography

In [ ]:
command=f"\n"
if do_bias == 1: 
    
    """
    Perform B1 field inhomogeneity correction for a DWI volume series

    Options for ANTs N4BiasFieldCorrection command - I use the default

    -ants_b [100,3] N4BiasFieldCorrection option -b: [initial mesh resolution in mm, spline order] This value is optimised for human adult data and needs to be adjusted for rodent data.
    -ants_c [1000,0.0] N4BiasFieldCorrection option -c: [numberOfIterations,convergenceThreshold]
    -ants_s 4 N4BiasFieldCorrection option -s: shrink-factor applied to spatial dimensions
    
    ref. Tustison, N.; Avants, B.; Cook, P.; Zheng, Y.; Egan, A.; Yushkevich, P. & Gee, J. N4ITK: Improved N3 Bias Correction. IEEE Transactions on Medical Imaging, 2010, 29, 1310-1320 

    The script assumes that eddy current correction has already been performed
    
    """
    for vp in vps:
        id = f"{SUBJTAG}{vp:02d}"   # id: standard subject ID (sub-XX)
        for session in sessions:    # Loop over sessions for this participant
            subjid = f"{id}_{session}"     # Standardized subject-session identifier 
            bash_script=os.path.join(f"{script}/DEWEY_v6/bash/", f"{subjid}_preprocessing.sh") 

            derivatives = os.path.join(prj_path, "derivatives", id, session)  #
            
            dwi_prep = os.path.join(derivatives, "dwi", "prep")  # Step path                
            eddy_path=os.path.join(derivatives, "dwi","eddy")

            biascorr=os.path.join(derivatives, "dwi","bias_corr")
            command+= f'if [ ! -d "{biascorr}" ]; then mkdir -p "{biascorr}"; fi\n'

            in_DWIs  = os.path.join(eddy_path,f"{subjid}_dwi_eddy_corrected.nii.gz")
            in_BVECs = os.path.join(eddy_path,f"{subjid}_dwi_eddy_corrected.eddy_rotated_bvecs")
            in_BVALs = os.path.join(dwi_prep,f"{subjid}_dir-PA-AP_part-mag_dwi.bval")

            mask=os.path.join(eddy_path, f"topup_out_unwarped_brain_mask.nii.gz")
            out_DWIs=os.path.join(biascorr,f"{subjid}_eddy-current_bias-corrected.nii.gz")
            
            bias=os.path.join(biascorr,  f"{subjid}_bias.nii.gz") # Estimated bias field image produced by ANTs
            residue_DwIs=os.path.join(biascorr,  f"{subjid}_residual.nii.gz") # Residual image computed as (eddy-corrected DWI − bias-corrected DWI)
            
            command+=f"dwibiascorrect ants {in_DWIs} {out_DWIs} -fslgrad {BVECs} {in_BVALs} -bias {bias} -force -debug\n" # Apply N4 bias field correction using ANTs while preserving gradient information
            command+=f"fslmaths {in_DWIs} -sub {out_DWIs} {residue_DwIs}\n" 

## Check preprocessing steps
-----------------------------------
Run this block to have a rough idea on how ther preprocessing is going, the output will be a matrix, one per preprocessing step and tels you how many steps/participants have been preprocessed.

In [ ]:
# =======================
# Configuration 
# =======================

vps = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43]#20, 21, 22, 25, 26, 27, 28, 29] #11, 12, 13, 14, 15, 16, 17]  8,23 and 24 are out # Set the subject number  0, 1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 25, 26, 27, 28, 29
sessions=["ses-02", "ses-01"] #same with potatoes THE SCRIPT ASSUMES THAT SESSIONS FOLDER ARE NAMED AS 'ses-0*'  #Number 1, sess1 checl 

do_prep         = False
do_denoise      = True
do_gibbsring    = False
do_topup        = False
do_eddy         = False
do_bias         = False
do_signaldrift  = True
do_dtifit       = True
do_kurtosis     = False
do_noddi        = False
do_freewater    = False
do_coreg        = False
do_smoothing    = False


# Map flag → folder keys 
STEP_FLAGS = {
    "prep":         do_prep,
    "denoise":      do_denoise,
    "gibbs_ring":   do_gibbsring,
    "topup":        do_topup,
    "eddy":         do_eddy,
    "bias_corr":    do_bias,
    "signal_drift": do_signaldrift,
    "Tensor":       do_dtifit,
    "Kurtosis":     do_kurtosis,
    "Noddi":        do_noddi,
    "Freewater":    do_freewater,
    "index2MNI":    do_coreg,
    "smooth":       do_smoothing,
}


# ============================================================
# Step registry: flag → (label, subfolders, output_builder)
# output_builder(subjid, ap) → filepath to check
# AP-dependent steps receive the ap letter; others receive None
# ============================================================
active_steps = [k for k, v in STEP_FLAGS.items() if v]

def build_outputs(subjid: str, ap: str) -> dict:
    sub_id  = subjid.split("_")[0]
    session = subjid.split("_")[1]
    derivatives   = os.path.join(prj_path, "derivatives", sub_id, session, "dwi")

    prep_path    = os.path.join(derivatives, "prep")
    denoise_path = os.path.join(derivatives, "denoise")
    gibbs_path   = os.path.join(derivatives, "gibbs_ring")
    topup_path   = os.path.join(derivatives, "topup")
    eddy_path    = os.path.join(derivatives, "eddy")
    drift_path   = os.path.join(derivatives, "signal_drift")
    dti_path     = os.path.join(derivatives, "index", "Tensor")
    kurt_path    = os.path.join(derivatives, "index", "Kurtosis")
    noddi_path   = os.path.join(derivatives, "index", "Noddi")
    fw_path      = os.path.join(derivatives, "index", "Freewater")
    coreg_path   = os.path.join(derivatives, "index2MNI")
    smooth_path  = os.path.join(derivatives, "index2MNI", "Smoothing")

    return { #Change names 
        "prep":         os.path.join(prep_path,    f"{subjid}_dir-PA-AP_part-mag_dwi.nii.gz"),
        "denoise":      os.path.join(denoise_path, f"{subjid}_dir-PA-AP_part-mag_dwi_denoised.nii.gz"),
        "gibbs_ring":   os.path.join(gibbs_path,   f"{subjid}_dir-PA-AP_part-mag_dwi_denoised_degibbs.nii.gz"),
        "topup":        os.path.join(topup_path,   f"topup_out_unwarped.nii"),
        "eddy":         os.path.join(eddy_path,    f"{subjid}_dwi_eddy_corrected.nii"), #  <--- U may remove nii/nii.gz
        "bias_corr":    os.path.join(prep_path,    f"{subjid}_bias_corrected.nii.gz"),
        "signal_drift": os.path.join(drift_path,   f"{subjid}_eddy-current_signal-drift.nii.gz"),
        "Tensor":       os.path.join(dti_path,     f"{subjid}_eddy-current_signal-drift_MD.nii"),
        "Kurtosis":     os.path.join(kurt_path,    f"{subjid}_MK.nii.gz"),
        "Noddi":        os.path.join(noddi_path,   f"{subjid}_NDI.nii.gz"),
        "Freewater":    os.path.join(fw_path,      f"{subjid}_FWF.nii.gz"),
        "index2MNI":    os.path.join(coreg_path,   f"{subjid}_eddy-current_signal-drift_MD2MNI.nii"),
        "smooth":       os.path.join(smooth_path,  f"{subjid}_eddy-current_signal-drift_MD2MNI6mm.nii"),
    }


# ============================================================
# Build matrix for one AP letter
# rows = sub-XX_ses-YY  |  cols = active steps
# 1 = exists, 0 = missing
# ============================================================
def build_matrix(ap: str):
    row_labels, matrix = [], []
    for vp in vps:
        id = f"sub-{vp:02d}"
        for session in sessions:
            subjid  = f"{id}_{session}"
            outputs = build_outputs(subjid, ap)
            row_labels.append(subjid)
            matrix.append([
                1 if os.path.exists(outputs[step]) else 0
                for step in active_steps
            ])
    return row_labels, np.array(matrix)

# ============================================================
# Plot one matrix panel
# ============================================================
CMAP = ListedColormap(["#9d5eb2","#489682"])   # red=missing, green=exists
NORM = plt.Normalize(vmin=0, vmax=1)

def plot_matrix(row_labels, matrix, title, ax):
    ax.imshow(matrix, aspect="auto", cmap=CMAP, norm=NORM, interpolation="nearest")

    # Grid
    ax.set_xticks(np.arange(-.5, len(active_steps), 1), minor=True)
    ax.set_yticks(np.arange(-.5, len(row_labels),   1), minor=True)
    ax.grid(which="minor", color="white", linewidth=1.5)
    ax.tick_params(which="minor", bottom=False, left=False)

    # Axis labels
    ax.set_xticks(range(len(active_steps)))
    ax.set_xticklabels(active_steps, rotation=40, ha="right", fontsize=9, fontweight="bold")
    ax.set_yticks(range(len(row_labels)))
    ax.set_yticklabels(row_labels, fontsize=7, fontfamily="monospace")
    ax.set_title(title, fontsize=12, fontweight="bold", pad=10)

    # Cell text
    for r in range(matrix.shape[0]):
        for c in range(matrix.shape[1]):
            txt = "" if matrix[r, c] == 1 else ""
            clr = "white"
            ax.text(c, r, txt, ha="center", va="center", fontsize=7, color=clr)

# ============================================================
# Render — one panel per AP letter
# ============================================================

n_rows  = len(vps) * len(sessions)
n_cols  = 1
fig_w   = max(8, len(active_steps) * 1.3) * n_cols
fig_h   = max(6, n_rows * 0.28)

fig, axes = plt.subplots(1, n_cols, figsize=(fig_w, fig_h))
if n_cols == 1:
    axes = [axes]

for ax in zip(axes):
    row_labels, mat = build_matrix
    plot_matrix(row_labels, mat, f"AP set — {ap}", ax)

legend = [
    mpatches.Patch(color="#489682", label="Exists"),
    mpatches.Patch(color="#9d5eb2", label="Missing"),
]
fig.legend(handles=legend, loc="lower center", ncol=2,
            fontsize=9, framealpha=0.9, bbox_to_anchor=(0.5, -0.04))

#plt.title("BIDS Pipeline — Output Checker", fontsize=14, fontweight="bold")
plt.show()



## DTIfit fitting and estimation 
----------------------------------------------

In [ ]:
command="\n"
do_dtifit=1
if do_dtifit == 1: 
    for vp in vps:
        id = f"{SUBJTAG}{vp:02d}"   # id: standard subject ID (sub-XX)
        for session in sessions:    # Loop over sessions for this participant
            subjid = f"{id}_{session}"     # Standardized subject-session identifier 
            bash_script = os.path.join(bash_dir, f"{subjid}_{BASH_SUFFIX}.sh")
            print(f"Compiling: {bash_script}")
            derivatives = os.path.join(prj_path, "derivatives", id, session)  #
            LOGsOut = os.path.join(LOGs_dir, f"{subjid}-LOGs", f"{subjid}-LOGs_{timestamp}.txt")

            dwi_prep = os.path.join(derivatives, "dwi", "prep")  # Step path                
            denoise_path=os.path.join(derivatives, "dwi","denoise") # Prep folder
            gibbs_path=os.path.join(derivatives, "dwi","gibbs_ring")
            eddy_path=os.path.join(derivatives, "dwi","eddy")
            signal_drift=os.path.join(derivatives, "dwi","signal_drift")

            ## index folders 
            index=os.path.join(derivatives, "dwi","index")
            Tensor=os.path.join(derivatives, "dwi","index", "Tensor")
            
            command+= f'if [ ! -d "{index}" ]; then mkdir -p "{index}"; fi\n'
            command+= f'if [ ! -d "{Tensor}" ]; then mkdir -p "{Tensor}"; fi\n'
                
            in_DWIs  = os.path.join(signal_drift,f"{subjid}_eddy-current_signal-drift_corr.nii.gz")
            in_BVECs = os.path.join(eddy_path,f"{subjid}_dwi_eddy_corrected.eddy_rotated_bvecs")
            in_BVALs = os.path.join(dwi_prep,f"{subjid}_dir-PA-AP_part-mag_dwi.bval")
            
            mask=os.path.join(eddy_path, f"topup_out_unwarped_brain_mask.nii.gz")
            
            in_DWIs_b1000  = os.path.join(index,f"{subjid}_eddy-current_signal-drift_corr_b1000.nii.gz")
            in_BVECs_b1000 = os.path.join(index,f"{subjid}_dwi_eddy_corrected_b1000.eddy_rotated_bvecs")
            in_BVALs_b1000 = os.path.join(index,f"{subjid}_dir-PA-AP_part-mag_dwi_denoised_degibbs_b1000.bval")

            out_DWIs = os.path.join(Tensor,f"{subjid}_eddy-current_signal-drift_corr")
            # Remove PAs volumes (they are the first 6 of the series)

            command+=f"echo '============================================'\n"
            command+=f"echo 'DTIfit:  {subjid} '\n"
            command+=f"echo '============================================'\n"
            command+=f"echo ' Input DWIs         : {in_DWIs_b1000}'\n"
            command+=f"echo ' Input BVECs        : {in_BVECs_b1000}'\n"
            command+=f"echo ' Input BVALs        : {in_BVALs_b1000}'\n"
            command+=f"echo ' Outpup DWIs        : {out_DWIs}'\n"
            command+=f"echo '============================================'\n"

            command+=f"dwiextract {in_DWIs} -shells 0,1000 {in_DWIs_b1000} -fslgrad {in_BVECs} {in_BVALs} -export_grad_fsl {in_BVECs_b1000} {in_BVALs_b1000} -force -debug \n" 
            command+=f"dtifit -k {in_DWIs_b1000} -o {out_DWIs} -m {mask} -r {in_BVECs_b1000} -b {in_BVALs_b1000} --save_tensor -V \n" 
        
            DWIs_PNG=os.path.join(Tensor,f"{subjid}_eddy-current_signal-drift_corr_MD.png")
            command+=f"python {script}/DEWEY_v6/HomeMadeSlicesDir.py -i {os.path.join(Tensor,f"{subjid}_eddy-current_signal-drift__corr_MD.nii")} -out {Tensor} -outpng {DWIs_PNG}\n\n " #Noise
            
            DWIs_PNG=os.path.join(Tensor,f"{subjid}_eddy-current_signal-drift__corr_FA.png")
            command+=f"python {script}/DEWEY_v6/HomeMadeSlicesDir.py -i {os.path.join(Tensor,f"{subjid}_eddy-current_signal-drift_corr_FA.nii")} -out {Tensor} -outpng {DWIs_PNG}\n\n " #Noise

            with open(bash_script, "a") as f: #Update LOGs file
                f.write(command)
            command=f"\n"

## DKI estimation 
----------------------------------------------
I will try to run on comfrey the script below jsut make it run in parallel

In [ ]:
command="\n"
if do_kurtosis==1:

    for vp in vps:
        id = f"{SUBJTAG}{vp:02d}"   # id: standard subject ID (sub-XX)
        for session in sessions:    # Loop over sessions for this participant
            subjid = f"{id}_{session}"     # Standardized subject-session identifier 
            bash_script = os.path.join(bash_dir, f"{subjid}_{BASH_SUFFIX}.sh")
            
            print(f"Compiling: {bash_script}")
            derivatives = os.path.join(prj_path, "derivatives", id, session)  #
            
            eddy_path=os.path.join(derivatives, "dwi","eddy")
            signal_drift=os.path.join(derivatives, "dwi","signal_drift")

            ## index folders 
            index=os.path.join(derivatives, "dwi","index")
            Kurtosis=os.path.join(derivatives, "dwi","index", "Kurtosis")
            
            os.makedirs(index, exist_ok=True)
            os.makedirs(Kurtosis, exist_ok=True)
                
            in_DWIs  = os.path.join(signal_drift,f"{subjid}_eddy-current_signal-drift_corr.nii.gz")
            in_BVECs = os.path.join(eddy_path,f"{subjid}_dwi_eddy_corrected.eddy_rotated_bvecs")
            in_BVALs = os.path.join(dwi_prep,f"{subjid}_dir-PA-AP_part-mag_dwi.bval")
            mask=os.path.join(eddy_path, f"topup_out_unwarped_brain_mask.nii")
        
            out_DWIs = os.path.join(Kurtosis,f"{subjid}_eddy-current_signal-drift_corr") 

            print (f"============================================")
            print (f" Kurtosis:  {subjid}")
            print (f"============================================")
            print (f" Input DWIs         : {in_DWIs}")
            print (f" Input BVECs        : {in_BVECs}")
            print (f" Input BVALs        : {in_BVALs}")
            print (f" Outpup DWIs        : {out_DWIs}")
            print (f"============================================")
        

            command+=f"python3.12 {script}/DEWEY_v6/DKI_compute.py -i {in_DWIs}  -m {mask} --bvec {in_BVECs}  --bval {in_BVALs} -o {out_DWIs} \n\n " #Noise

            with open(bash_script, "a") as f: #Update LOGs file
                f.write(command)
            command=f"\n"

            

## NODDI 
---------------------------------------------
This is not on bash, it uses ython libraries, so i decided to run it on jupyter notebook - The script can be easily converted to .py script in case

In [ ]:
## AMICO (Noddi) -- Settings are based on SK's noddi code
import amico
amico.setup()
ae = amico.Evaluation()
ae.set_config('doComputeNRMSE', True) #'doDebiasSignal', True)              #'doSaveModulatedMaps', True)
ae.set_config('doSaveModulatedMaps', True)

# NODDI  params
# The model is setted on: 
# dPar ==> intrinsic parallel diffusivity; 0.0017 for WM but 0.0011 for GM! aka. Parallel diffusivity [mm^2/s].
# dIso ==> Maybe should higher in GM, aka  Isotropic diffusivity [mm^2/s]. 
# IC_VFs ==> Intra-cellular volume fractions. (default is np.linspace(0.1, 0.99, 12))
# IC_ODs ==> Intra-cellular orientation dispersions. (default is np.hstack((np.array([0.03, 0.06]), np.linspace(0.09, 0.99, 10))))

# NODDI diffusion parameters (defined once as constants)
# dPar : intrinsic parallel diffusivity [mm²/s]  (WM≈0.0017, GM≈0.0011)
# dIso : isotropic (CSF) diffusivity [mm²/s]
DPAR = 0.0011
DISO = 0.003


# Initialise AMICO once, outside all loops
command="\n"

for vp in vps:
    id = f"{SUBJTAG}{vp:02d}"   # id: standard subject ID (sub-XX)
    for session in sessions:    # Loop over sessions for this participant
        subjid = f"{id}_{session}"     # Standardized subject-session identifier 

        derivatives = os.path.join(prj_path, "derivatives", id, session)  #
        LOGsOut = os.path.join(LOGs_dir, f"{subjid}-LOGs", f"{subjid}-LOGs_{timestamp}.txt")

        dwi_prep = os.path.join(derivatives, "dwi", "prep")  # Step path                
        denoise_path=os.path.join(derivatives, "dwi","denoise") # Prep folder
        gibbs_path=os.path.join(derivatives, "dwi","gibbs_ring")
        eddy_path=os.path.join(derivatives, "dwi","eddy")
        signal_drift=os.path.join(derivatives, "dwi","signal_drift")

        ## index folders 
        index=os.path.join(derivatives, "dwi","index")
        Noddi=os.path.join(derivatives, "dwi","index", "Noddi")
        
        os.makedirs(index, exist_ok=True)
        os.makedirs(Noddi, exist_ok=True)

            
        in_DWIs  = os.path.join(signal_drift,f"{subjid}_eddy-current_signal-drift_corr.nii.gz")
        in_BVECs = os.path.join(eddy_path,f"{subjid}_dwi_eddy_corrected.eddy_rotated_bvecs")
        in_BVALs = os.path.join(dwi_prep,f"{subjid}_dir-PA-AP_part-mag_dwi.bval")
        mask=os.path.join(eddy_path, f"topup_out_unwarped_brain_mask.nii")
    
        out_DWIs = os.path.join(Noddi,f"{subjid}_eddy-current_signal-drift_corr")
        schemefile = os.path.join(Noddi, 'NODDI_protocol.scheme')


        print (f"============================================")
        print (f" NODDI:  {subjid} ")
        print (f"============================================")
        print (f" Input DWIs         : {in_DWIs}")
        print (f" Input BVECs        : {in_BVECs}")
        print (f" Input BVALs        : {in_BVALs}")
        print (f" Outpup DWIs        : {out_DWIs}")
        print (f"============================================")
    

        command+=f"python3.12 NODDI_compute.py -i {in_DWIs} -m {mask} --bvec {in_BVECs} --bval {in_BVALs} -o {out_DWIs} --dPar {DPAR} --dIso {DISO} --no-modulated \n" 
    
    with open(bash_script, "a") as f: #Update LOGs file
        f.write(command)
    command=f"\n"

            

## Coregistration and Smoothing 
--------------------------------------------------

In [ ]:
METRICS = {
    "Tensor":    ["V1", "V3"],# ["FA", "MD"],
    "Kurtosis":  ["MK", "MDk", "FAk", "MSD", "MSK"],
    "Noddi":     ["NDI", "ODI", "FWF"],
    "FreeWater": ["fw", "MDfw", "FAfw"],
}

# File-extension per model (FSL requires .nii for Tensor output)
MODEL_EXT = {
    "Tensor":    ".nii",
    "Kurtosis":  ".nii.gz",
    "Noddi":     ".nii.gz",
    "FreeWater": ".nii.gz",
}


model= "Kurtosis"

  
if do_coreg == 1:

    for vp in vps:
        id = f"{SUBJTAG}{vp:02d}"   # id: standard subject ID (sub-XX)

        for session in sessions:
            subjid      = f"{id}_{session}"
            derivatives = os.path.join(prj_path, "derivatives", id, session)
            anat        = os.path.join(prj_path, "derivatives", id, "ses-02", "anat")

            # ── Output folders ────────────────────────────────────────────────
            Tensor    = os.path.join(derivatives, "dwi", "index", "Tensor")
            Noddi     = os.path.join(derivatives, "dwi", "index", "Noddi")
            Kurtosis  = os.path.join(derivatives, "dwi", "index", "Kurtosis")
            FreeWater = os.path.join(derivatives, "dwi", "index", "FreeWater")
            coreg     = os.path.join(derivatives, "dwi", "coreg")
            index2MNI = os.path.join(derivatives, "dwi", "index2MNI")

            # ── Bash script setup ─────────────────────────────────────────────
            bash_script = os.path.join(bash_dir, f"{subjid}_{BASH_SUFFIX}.sh")
            print(f"Compiling: {bash_script}")

            command  = "#!/bin/bash\nset -euo pipefail\n\n"
            command += f'mkdir -p "{coreg}" "{index2MNI}"\n\n'

            # ── Anatomy / MNI references (defined once per session) ───────────
            T1w_brain   = os.path.join(anat, "brain-extraction",f"{id}_ses-02_desc-brain_t1w.nii.gz")

            # FSL MNI152
            t1w2MNI     = os.path.join(anat, "mni-registration",f"{id}_ses-02_desc-nonlin1warp_xfm.nii.gz")
            t1w2MNI_lin = os.path.join(anat, "mni-registration",f"{id}_ses-02_desc-nonlin0genericaffine_xfm.mat")

            # HCPex
            t1w2HCPex     = os.path.join(anat, "mni-registration",f"{id}_ses-02_HCPex-nonlin1Warp.nii.gz")
            t1w2HCPex_lin = os.path.join(anat, "mni-registration",f"{id}_ses-02_HCPex-nonlin0GenericAffine.mat")


            FA = os.path.join(Tensor,f"{subjid}_eddy-current_signal-drift_corr_FA.nii")
            FA2t1w_matrix  = os.path.join(coreg, f"{subjid}_FA2t1w_0GenericAffine.mat")
            FA2t1w_prefix  = os.path.join(coreg, f"{subjid}_FA2t1w_")

            # ── FA → T1w registration (skip if matrix already exists) ─────
            command += (
                f'if [ ! -f "{FA2t1w_matrix}" ]; then\n'          # -f: file, not -d: dir
                f'    antsRegistrationSyN.sh -d 3 -f {T1w_brain} -m {FA} -o {FA2t1w_prefix} -t a -n 10\n'
                f'else\n'
                f'    echo "Skipping coreg — matrix exists: {FA2t1w_matrix}"\n'
                f'fi\n\n'
            )


            MODEL_DIRS = {
                "Tensor":    Tensor,
                "Kurtosis":  Kurtosis,
                "Noddi":     Noddi,
                "FreeWater": FreeWater,
            }

            metrics=METRICS[model]#["NDI", "ODI", "FWF"]
            mdir = MODEL_DIRS[model]
            ext = MODEL_EXT[model]

            command+=f"echo '=================================================='\n"
            command+=f"echo 'Coregistration to MNI and HCPex template:  {model} '\n"
            command+=f"echo '=================================================='\n"


            for metric in metrics:
                base_name = f"{subjid}_eddy-current_signal-drift_corr_{metric}"
                src      = os.path.join(mdir,      f"{subjid}_eddy-current_signal-drift_corr_{metric}{ext}")
                out_mni  = os.path.join(index2MNI, f"{subjid}_eddy-current_signal-drift_corr_{metric}2MNI{ext}")
                out_hcp  = os.path.join(index2MNI, f"{subjid}_eddy-current_signal-drift_corr_{metric}2MNI-HCPex{ext}")

                command += (
                    f'if [ -f "{src}" ]; then\n'
                    f'    antsApplyTransforms -d 3 -e 0 -i {src} -n BSpline --transform {t1w2MNI} --transform {t1w2MNI_lin} --transform {FA2t1w_matrix} -r {MNI_BRAIN} -o {out_mni} -v\n'
                    f'    antsApplyTransforms -d 3 -e 0 -i {src} -n BSpline --transform {t1w2HCPex} --transform {t1w2HCPex_lin} --transform {FA2t1w_matrix} -r {HCPEX_BRAIN} -o {out_hcp} -v\n'
                    f'fi\n\n'
                    )
        # ── Write and submit bash script ──────────────────────────────────
        with open(bash_script, "a") as f: #Update LOGs file
            f.write(command)
        command=f"\n"



In [ ]:
l2fwhm      = [1.69865806, 2.54798709]   # ≈ 4 mm / 6 mm FWHM in MNI space1.69865806,"4mm",
kernel_sizes = ["4mm","6mm"]
model="Kurtosis"
METRICS = {
    "Tensor":    ["V1", "V3"],# ["FA", "MD"],
    "Kurtosis":  ["MK", "MDk", "FAk", "MSD", "MSK"],
    "Noddi":     ["NDI", "ODI", "FWF"],
    "FreeWater": ["fw", "MDfw", "FAfw"],
}

# File-extension per model (FSL requires .nii for Tensor output)
MODEL_EXT = {
    "Tensor":    ".nii",
    "Kurtosis":  ".nii.gz",
    "Noddi":     ".nii.gz",
    "FreeWater": ".nii.gz",
}


def Smoothing(dwi, kernel, mni_mask, smooth_dir, out_path):
    """
    Return FSL commands that perform within-mask Gaussian smoothing.

    The two-step divide-by-smoothed-mask approach corrects for edge
    attenuation at the brain boundary.

    Parameters
    ----------
    dwi        : str  – input NIfTI (already in MNI space)
    kernel     : float – sigma in mm passed to fslmaths -s
    mni_mask   : str  – binary brain mask in the same space
    smooth_dir : str  – directory used for temporary files
    out_path   : str  – full path of the smoothed output image
    tag        : str  – unique string to avoid tmp-file collisions
    """
    tmp1 = os.path.join(smooth_dir, f"_tmp1.nii.gz")
    tmp2 = os.path.join(smooth_dir, f"_tmp2.nii.gz")

    cmd =f" echo 'Smoothing → {out_path}'\n"
    cmd += (
        f'fslmaths {dwi} -mas {mni_mask} -s {kernel} -mas {mni_mask} {tmp1}\n'
        f'fslmaths {mni_mask} -s {kernel} -mas {mni_mask} {tmp2}\n'
        f'fslmaths {tmp1} -div {tmp2} {out_path}\n'
        f'rm -f {tmp1} {tmp2}\n'  # clean up intermediates
    )
    return cmd
    
if do_smoothing == 1:
    for vp in vps:
        id_ = f"sub-{vp:02d}"

        for session in sessions:
            subjid      = f"{id_}_{session}"
            derivatives = os.path.join(prj_path, "derivatives", id_, session)
            anat        = os.path.join(prj_path, "derivatives", id_, "ses-02", "anat")

            # ── Output folders ────────────────────────────────────────────────
            Tensor    = os.path.join(derivatives, "dwi", "index", "Tensor")
            Noddi     = os.path.join(derivatives, "dwi", "index", "Noddi")
            Kurtosis  = os.path.join(derivatives, "dwi", "index", "Kurtosis")
            FreeWater = os.path.join(derivatives, "dwi", "index", "FreeWater")
            coreg     = os.path.join(derivatives, "dwi", "coreg")
            index2MNI = os.path.join(derivatives, "dwi", "index2MNI")
            smooth    = os.path.join(derivatives, "dwi", "index2MNI", "Smoothing")

            # ── Bash script setup ─────────────────────────────────────────────
            bash_script = os.path.join(bash_dir, f"{subjid}_{BASH_SUFFIX}.sh")
            print(f"Compiling: {bash_script}")

            command  = "\n"
            command += f'mkdir -p "{smooth}" \n\n'
            
            MODEL_DIRS = {
                    "Tensor":    Tensor,
                    "Kurtosis":  Kurtosis,
                    "Noddi":     Noddi,
                    "FreeWater": FreeWater,
                }

            #for model, metrics in METRICS.items():

            metrics=METRICS[model]#["NDI", "ODI", "FWF"]
            mdir = MODEL_DIRS[model]
            ext = MODEL_EXT[model]

            command+=f"echo '=================================================='\n"
            command+=f"echo 'Smoothing to 4 and 6mm:  {model} '\n"
            command+=f"echo '=================================================='\n"


            #   ext = MODEL_EXT[model]
            for metric in metrics:
                for i,sigma in enumerate(l2fwhm):
                    
                    in_mni  = os.path.join(index2MNI, f"{subjid}_eddy-current_signal-drift_{AP}_corr_{metric}2MNI{ext}")
                    out_mni  = os.path.join(smooth, f"{subjid}_eddy-current_signal-drift_{AP}_corr_{metric}2MNI{kernel_sizes[i]}{ext}")
                    command+=Smoothing(in_mni, sigma, MNI_MASK, smooth, out_mni)

                    in_hcp  = os.path.join(index2MNI, f"{subjid}_eddy-current_signal-drift_{AP}_corr_{metric}2MNI-HCPex{ext}")
                    out_hcp  = os.path.join(smooth, f"{subjid}_eddy-current_signal-drift_{AP}_corr_{metric}2MNI-HCPex{kernel_sizes[i]}{ext}")
                    command+=Smoothing(in_hcp, sigma, HCPEX_MASK, smooth, out_hcp)
        
            # ── Write and submit bash script ────────────────────────────────── smooth
            with open(bash_script, "a") as f: #Update LOGs file
                f.write(command)
            command=f"\n"

# RUN bash script! 
--------------------------------
run all the script i created and potentially, run them in parallel

In [ ]:
import os
import subprocess
from concurrent.futures import ProcessPoolExecutor, as_completed

# ============================================================
# Configuration
# ============================================================

N_PPOOL  = 2 # --> Numb. process at the same tiem
#vps =[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43]#20, 21, 22, 25, 26, 27, 28, 29] #11, 12, 13, 14, 15, 16, 17]  8,23 and 24 are out # Set the subject number  0, 1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 25, 26, 27, 28, 29

# ============================================================
# Build job list -- this is claude not me
# ============================================================
jobs = [(vp, session) for vp in vps for session in sessions]

print(f"\n Starting | {len(jobs)} jobs | {N_PPOOL} parallel\n")

# ============================================================
# Run in parallel
# ============================================================
results = []

with ProcessPoolExecutor(max_workers=N_PPOOL) as executor:
    futures = {
        executor.submit(
            subprocess.run,
            os.path.join(script, "DEWEY_v6", "bash", f"sub-{vp:02d}_{session}_DKI_Coreg_smoothing.sh"), #                 bash_script = os.path.join(bash_dir, f"{subjid}_22Apr_preprocessing.sh")

            shell=True
        ): (vp, session)
        for vp, session in jobs
    }

    for future in as_completed(futures):
        vp, session = futures[future]
        subjid      = f"sub-{vp:02d}_{session}"
        try:
            result = future.result()
            if result.returncode == 0:
                print(f"  ✅ Done:   {subjid}")
                results.append((subjid, "done"))
            else:
                print(f"  ❌ Failed: {subjid} (exit code {result.returncode})")
                results.append((subjid, "failed"))
        except Exception as e:
            print(f"  💥 Error:  {subjid} → {e}")
            results.append((subjid, "error"))

# ============================================================
# Summary - still claude , but its' usefull
# ============================================================
print("\n========== SUMMARY ==========")
icons = {"done": "✅", "failed": "❌", "error": "💥"}
for subjid, status in sorted(results):
    print(f"  {icons.get(status, '?')}  {subjid}: {status}")
print("==============================\n")